# Sentinel-1 H/Alpha Decomposition

**Source script:** `grdl/example/image_processing/sar/sentinel1_halpha_decomposition.py`

This notebook demonstrates dual-pol H/Alpha decomposition on Sentinel-1 IW SLC data:

1. **Read Sentinel-1 IW SLC** — VV/VH dual-pol from .SAFE directory
2. **H/Alpha decomposition** — entropy (H) and alpha angle (α) from coherency matrix
3. **RGB composite** — visualize scattering randomness and mechanism

## H/Alpha Decomposition Theory

The H/Alpha decomposition is based on the eigenvalue decomposition of the 2×2 coherency matrix:

$$
T_2 = \langle k_2 k_2^H \rangle
$$

where the Pauli target vector for dual-pol (co-pol, cross-pol) is:

$$
k_2 = \frac{1}{\sqrt{2}} \begin{bmatrix} S_{vv} + S_{hh} \\ S_{vv} - S_{hh} \end{bmatrix}
$$

The coherency matrix eigenvalues $\lambda_1 \geq \lambda_2$ yield:

**Entropy (H)** — scattering randomness:
$$
H = -\sum_{i=1}^{2} p_i \log_2(p_i), \quad p_i = \frac{\lambda_i}{\lambda_1 + \lambda_2}
$$

- $H = 0$ → deterministic (single dominant scatterer)
- $H = 1$ → random (distributed scatterers, depolarization)

**Alpha angle (α)** — scattering mechanism:
$$
\alpha = \sum_{i=1}^{2} p_i \alpha_i
$$

where $\alpha_i = \cos^{-1}(|e_{i1}|)$ from eigenvector $e_i$.

- $\alpha \approx 0°$ → surface scattering (Bragg, smooth surfaces)
- $\alpha \approx 45°$ → volume scattering (vegetation, random media)
- $\alpha \approx 90°$ → double-bounce (urban, oriented structures)

**Anisotropy (A)** — secondary scattering component:
$$
A = \frac{\lambda_1 - \lambda_2}{\lambda_1 + \lambda_2}
$$

**RGB channel mapping**:
- **R** (red) = Span (total power, dB-stretched)
- **G** (green) = Entropy $H$ (scattering randomness)
- **B** (blue) = Alpha $\alpha / 90$ (scattering mechanism normalized to [0,1])

## Data Setup

Set `SAFE_DIR` in Cell 3 to point to a Sentinel-1 IW SLC `.SAFE` directory.
Requires dual-pol data (VV/VH or HH/HV).

In [ ]:
"""GRDL Notebook Preamble — Environment validation and autoreload."""
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except (NameError, AttributeError):
    pass

try:
    import grdl
except ImportError as exc:
    raise ImportError(
        "\n"
        "═══════════════════════════════════════════════════════════════\n"
        "  grdl is NOT installed in this Python environment.\n"
        "  Install with:  pip install -e '.[all]'\n"
        "  from the grdl repository root.\n"
        "═══════════════════════════════════════════════════════════════\n"
    ) from exc

print(f"grdl {grdl.__version__} — ready")

In [ ]:
from pathlib import Path
import numpy as np

from grdl.IO import get_reader
from grdl.image_processing.decomposition import DualPolHAlpha

# ════════════════════════════════════════════════════════════════════
# USER CONFIGURATION — Set your Sentinel-1 .SAFE path here
# ════════════════════════════════════════════════════════════════════
SAFE_DIR = Path("/path/to/your/S1A_IW_SLC__1SDV_*.SAFE")  # ← CHANGE THIS
SWATH = 'IW1'             # IW swath to process (IW1, IW2, or IW3)
CO_POL = 'VV'             # Co-polarization channel
CROSS_POL = 'VH'          # Cross-polarization channel
WINDOW_SIZE = 7           # Spatial averaging window for coherency matrix (odd, >= 3)
CHIP_SIZE = None          # Read center chip (pixels), or None for full scene

assert SAFE_DIR.exists(), f"SAFE directory not found: {SAFE_DIR}"
print(f"Target: {SAFE_DIR.name}")

## 1. Open Sentinel-1 Reader

Sentinel-1 IW SLC data is stored in a `.SAFE` directory with separate TIFF files
per polarization. The reader constructs a CYX (channel, row, column) cube
when `polarizations` is specified as a list.

In [ ]:
pols = [CO_POL.upper(), CROSS_POL.upper()]

print(f"Opening {SAFE_DIR.name}  swath={SWATH}  pols={pols} ...")
with get_reader('sentinel1-slc', SAFE_DIR) as reader:
    # Configure reader for multi-pol cube mode
    reader.swath = SWATH
    reader.polarizations = pols  # builds CYX cube: (2, rows, cols)
    
    meta = reader.metadata
    rows, cols = meta.rows, meta.cols
    
    available_pols = [cm.polarization for cm in meta.channel_metadata] if meta.channel_metadata else []
    print(f"  Available polarizations: {available_pols}")
    print(f"  Image size: {rows} × {cols} px")
    print(f"  Bursts: {meta.num_bursts}")
    print(f"  Axis order: {meta.axis_order}")
    
    # Read data (chip or full scene)
    if CHIP_SIZE is not None:
        r0 = max(0, rows // 2 - CHIP_SIZE // 2)
        c0 = max(0, cols // 2 - CHIP_SIZE // 2)
        r1 = min(rows, r0 + CHIP_SIZE)
        c1 = min(cols, c0 + CHIP_SIZE)
        print(f"Reading chip [{r0}:{r1}, {c0}:{c1}] ...")
        cube = reader.read_chip(r0, r1, c0, c1)  # (2, chip_rows, chip_cols)
    else:
        print("Reading full scene ...")
        cube = reader.read_full()  # (2, rows, cols)

print(f"\nCube shape: {cube.shape}, dtype: {cube.dtype}")

## 2. H/Alpha Decomposition

The `DualPolHAlpha` processor computes the coherency matrix eigendecomposition
and derives entropy, alpha, anisotropy, and span.

**`window_size`** controls spatial averaging for coherency matrix estimation:
- Smaller → preserves edges, noisier estimates
- Larger → smoother, better statistics, blurs features

In [ ]:
import gc

print(f"Running DualPolHAlpha (window_size={WINDOW_SIZE}) ...")
ha = DualPolHAlpha(window_size=WINDOW_SIZE)

# Channel 0 = co-pol, channel 1 = cross-pol
components, _ = ha.execute(meta, cube)
# components: dict with keys 'entropy', 'alpha', 'anisotropy', 'span'

print(f"\nComponent ranges:")
print(f"  Entropy    H : [{components['entropy'].min():.3f}, {components['entropy'].max():.3f}]")
print(f"  Alpha      α : [{components['alpha'].min():.1f}°, {components['alpha'].max():.1f}°]")
print(f"  Anisotropy A : [{components['anisotropy'].min():.3f}, {components['anisotropy'].max():.3f}]")
print(f"  Span         : [{components['span'].min():.2e}, {components['span'].max():.2e}]")

# Free dual-pol cube (no longer needed)
del cube
gc.collect()

## 3. Build RGB Composite

The H/Alpha processor provides a `to_rgb()` method that creates a composite:
- **R** = Span (dB-stretched total power)
- **G** = Entropy (scattering randomness)
- **B** = Alpha / 90 (scattering mechanism normalized)

In [ ]:
print("Building RGB composite ...")
rgb_float, rgb_meta = ha.to_rgb(components)
# rgb_float: (3, rows, cols) float32 in [0, 1]

print(f"RGB float shape: {rgb_float.shape}")
print(f"RGB range: [{rgb_float.min():.3f}, {rgb_float.max():.3f}]")

# Convert to uint8 for display (HWC format)
rgb_uint8 = (np.clip(rgb_float.transpose(1, 2, 0), 0.0, 1.0) * 255).astype(np.uint8)
print(f"RGB uint8 shape: {rgb_uint8.shape}")

# Free components and float RGB (no longer needed for visualization)
del components, rgb_float
gc.collect()

## 4. Visualization with napari

Display the H/Alpha RGB composite and individual parameter maps.

**Interpretation guide**:

| Feature | Low H, Low α | Low H, High α | High H, Low α | High H, High α |
|---------|--------------|---------------|---------------|----------------|
| **Type** | Smooth surface | Oriented double-bounce | Distributed surface | Distributed volume |
| **Examples** | Water, roads | Buildings, fences | Rough terrain | Forest canopy |
| **Color** | Dark blue | Dark purple | Cyan | Green-cyan |

**Cloude-Pottier zones**:
- **Zone 1-3**: Low entropy (deterministic scattering)
- **Zone 4-6**: Medium entropy (moderate randomness)
- **Zone 7-9**: High entropy (highly random / depolarized)

In [ ]:
import napari

viewer = napari.Viewer(title="Sentinel-1 H/Alpha Decomposition")

# H/Alpha RGB composite
viewer.add_image(
    rgb_uint8,
    name="H/Alpha RGB (R=Span G=H B=α)",
    rgb=True,
)

# Span (dB)
span_db = 10.0 * np.log10(np.maximum(components['span'], np.finfo(float).tiny))
p2, p98 = np.percentile(span_db, [2, 98])
span_norm = np.clip((span_db - p2) / (p98 - p2 + 1e-12), 0, 1)
viewer.add_image(
    span_norm,
    name="Span (dB)",
    colormap="gray",
    visible=False,
)

# Entropy
viewer.add_image(
    components['entropy'],
    name="Entropy H",
    colormap="hot",
    contrast_limits=[0, 1],
    visible=False,
)

# Alpha
viewer.add_image(
    components['alpha'],
    name="Alpha α (degrees)",
    colormap="jet",
    contrast_limits=[0, 90],
    visible=False,
)

# Anisotropy
viewer.add_image(
    components['anisotropy'],
    name="Anisotropy A",
    colormap="cool",
    contrast_limits=[0, 1],
    visible=False,
)

print(f"napari viewer opened with {len(viewer.layers)} layers")
print("Toggle individual parameter layers to explore H/Alpha components.")
print("\nParameter interpretation:")
print("  Entropy H=0   → Deterministic scattering (single dominant mechanism)")
print("  Entropy H=1   → Random scattering (depolarized, distributed)")
print("  Alpha α=0°    → Surface scattering (Bragg, smooth)")
print("  Alpha α=45°   → Volume scattering (vegetation, random media)")
print("  Alpha α=90°   → Double-bounce (urban, vertical structures)")
print("  Anisotropy A  → Relative importance of 2nd eigenvalue")

## 5. Cloude-Pottier Classification

Partition the H-α plane into 9 zones for unsupervised classification.

In [ ]:
# Define H-alpha zone boundaries
h_low, h_high = 0.5, 0.9
a_low, a_high = 40.0, 50.0

# Zone classification
zones = np.zeros_like(components['entropy'], dtype=np.uint8)

# Low entropy (H < 0.5)
mask_h_low = components['entropy'] < h_low
zones[mask_h_low & (components['alpha'] < a_low)] = 1   # Zone 1: surface
zones[mask_h_low & (components['alpha'] >= a_low) & (components['alpha'] < a_high)] = 2  # Zone 2: dipole
zones[mask_h_low & (components['alpha'] >= a_high)] = 3  # Zone 3: double-bounce

# Medium entropy (0.5 <= H < 0.9)
mask_h_med = (components['entropy'] >= h_low) & (components['entropy'] < h_high)
zones[mask_h_med & (components['alpha'] < a_low)] = 4   # Zone 4: anisotropic surface
zones[mask_h_med & (components['alpha'] >= a_low) & (components['alpha'] < a_high)] = 5  # Zone 5: vegetation
zones[mask_h_med & (components['alpha'] >= a_high)] = 6  # Zone 6: anisotropic double-bounce

# High entropy (H >= 0.9)
mask_h_high = components['entropy'] >= h_high
zones[mask_h_high & (components['alpha'] < a_low)] = 7   # Zone 7: random surface
zones[mask_h_high & (components['alpha'] >= a_low) & (components['alpha'] < a_high)] = 8  # Zone 8: random volume
zones[mask_h_high & (components['alpha'] >= a_high)] = 9  # Zone 9: random double-bounce

# Count pixels per zone
zone_labels = [
    "Background",
    "Z1: Low entropy surface",
    "Z2: Low entropy dipole",
    "Z3: Low entropy double-bounce",
    "Z4: Medium entropy surface",
    "Z5: Medium entropy vegetation",
    "Z6: Medium entropy double-bounce",
    "Z7: High entropy surface",
    "Z8: High entropy volume",
    "Z9: High entropy double-bounce",
]

print("\n═══ Cloude-Pottier Zone Distribution ═══")
for z in range(1, 10):
    count = np.sum(zones == z)
    frac = 100 * count / zones.size
    print(f"  {zone_labels[z]:40s}: {frac:5.2f}% ({count:,} px)")

# Add zones as labels layer in napari
viewer.add_labels(
    zones,
    name="Cloude-Pottier Zones",
    opacity=0.5,
)

print("\nAdded Cloude-Pottier zone classification to napari viewer.")

---

## Summary

This notebook replaces `grdl/example/image_processing/sar/sentinel1_halpha_decomposition.py`.

Key patterns:
- `get_reader('sentinel1-slc', path)` — factory pattern for Sentinel-1 .SAFE
- Multi-pol cube mode: `reader.polarizations = [co, cross]` → CYX (2, rows, cols)
- `DualPolHAlpha` — eigendecomposition of 2×2 coherency matrix
- `ha.to_rgb()` — automatic RGB composite generation
- napari RGB + parameter layers + Cloude-Pottier classification

**Physical interpretation**:
- **Entropy** measures scattering randomness (0=deterministic, 1=depolarized)
- **Alpha** identifies scattering mechanism (0°=surface, 45°=volume, 90°=double-bounce)
- **Anisotropy** quantifies secondary scattering component importance
- **Cloude-Pottier zones** provide unsupervised classification based on H-α plane

**Requires**: Sentinel-1 IW SLC dual-pol data (VV/VH or HH/HV)